# Module 10: Interview Simulation

**Purpose:** Simulate a 45-minute technical coding assessment on diffusion models.

This module contains timed coding exercises, verbal explanation prompts, a bug-finding challenge,
and a system design discussion. Each exercise mirrors the format and difficulty of a real interview.

**How to use this module:**
1. Set a timer for each exercise (time limits are noted).
2. Attempt the exercise without looking at the solution.
3. After time is up, review the solution and debrief.
4. For verbal questions, speak your answer aloud before reading the model answer.

**Prerequisites:** Modules 0-9 (PyTorch fundamentals through advanced topics).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, matplotlib.pyplot as plt, math
from tqdm.auto import tqdm
torch.manual_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---

## Exercise 10.1: 2D Point Cloud Diffusion (45 minutes)

**Set your timer to 45 minutes now.**

**Prompt:** Implement a complete diffusion model that learns to generate points from a 2D spiral distribution. You must implement the forward noising process, a denoiser network, training, and sampling.

**Evaluation Criteria:**
- Correct linear noise schedule with proper alpha/alpha_bar computation
- Sinusoidal timestep embedding in the denoiser
- Correct forward process: $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \epsilon$
- Correct DDPM reverse sampling loop
- Clean, readable code with shape comments

**Expected Deliverables:**
1. `make_spiral()` function returning a batch of 2D points
2. Noise schedule tensors (`betas`, `alphas`, `alpha_bars`)
3. MLP denoiser with sinusoidal time embedding
4. Training loop (5000 steps)
5. DDPM sampling function
6. Scatter plot comparing generated vs. training data

In [ ]:
def make_spiral(n_points: int = 2000, noise: float = 0.3) -> torch.Tensor:
    """Generate a 2D spiral point cloud.

    Args:
        n_points: Number of points to generate.
        noise: Standard deviation of Gaussian noise added to each point.

    Returns:
        Tensor of shape (n_points, 2) with spiral coordinates.
    """
    t = torch.linspace(0, 4 * math.pi, n_points)                 # (n_points,)
    x = t * torch.cos(t) + noise * torch.randn(n_points)         # (n_points,)
    y = t * torch.sin(t) + noise * torch.randn(n_points)         # (n_points,)
    data = torch.stack([x, y], dim=1)                             # (n_points, 2)
    data = (data - data.mean(0)) / data.std(0)                    # normalize to ~N(0,1)
    return data

# Quick check
spiral_data = make_spiral()
plt.figure(figsize=(5, 5))
plt.scatter(spiral_data[:, 0].numpy(), spiral_data[:, 1].numpy(), s=2, alpha=0.5)
plt.title("Training Data: 2D Spiral")
plt.axis("equal")
plt.show()
print(f"spiral_data.shape = {spiral_data.shape}")  # (2000, 2)

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Exercise 10.1: Complete 2D Point Cloud Diffusion

torch.manual_seed(42)

# ---------- Noise schedule ----------
T = 300
betas = torch.linspace(1e-4, 0.02, T).to(device)                    # (T,)
alphas = 1.0 - betas                                                  # (T,)
alpha_bars = torch.cumprod(alphas, dim=0)                             # (T,)


# ---------- Sinusoidal timestep embedding ----------
class SinusoidalEmbedding(nn.Module):
    """Maps scalar timestep to a sinusoidal positional embedding vector."""

    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: (B,) integer timesteps.
        Returns:
            (B, embed_dim) sinusoidal embedding.
        """
        half = self.embed_dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device) / half
        )                                                              # (half,)
        args = t[:, None].float() * freqs[None, :]                     # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)   # (B, embed_dim)


# ---------- MLP Denoiser ----------
class MLPDenoiser(nn.Module):
    """Simple MLP that predicts noise given (x_t, t)."""

    def __init__(self, data_dim: int = 2, embed_dim: int = 64, hidden: int = 256):
        super().__init__()
        self.time_embed = SinusoidalEmbedding(embed_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + embed_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 2) noisy data.
            t: (B,) integer timesteps.
        Returns:
            (B, 2) predicted noise.
        """
        t_emb = self.time_embed(t)                       # (B, embed_dim)
        inp = torch.cat([x, t_emb], dim=-1)              # (B, 2 + embed_dim)
        return self.net(inp)                              # (B, 2)


# ---------- Training ----------
data = make_spiral(n_points=2000).to(device)              # (2000, 2)
model = MLPDenoiser().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for step in tqdm(range(5000), desc="Training"):
    # Sample batch from data
    idx = torch.randint(0, len(data), (256,))
    x_0 = data[idx]                                       # (256, 2)

    # Sample random timesteps
    t = torch.randint(0, T, (256,), device=device)        # (256,)

    # Sample noise
    eps = torch.randn_like(x_0)                           # (256, 2)

    # Forward process: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
    ab_t = alpha_bars[t][:, None]                         # (256, 1)
    x_t = torch.sqrt(ab_t) * x_0 + torch.sqrt(1 - ab_t) * eps  # (256, 2)

    # Predict noise
    eps_pred = model(x_t, t)                              # (256, 2)

    # MSE loss
    loss = F.mse_loss(eps_pred, eps)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(losses, alpha=0.3)
plt.plot(np.convolve(losses, np.ones(100)/100, mode='valid'), color='red')
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()


# ---------- DDPM Sampling ----------
@torch.no_grad()
def ddpm_sample(model: nn.Module, n_samples: int = 2000) -> torch.Tensor:
    """Generate samples via the full DDPM reverse process.

    Args:
        model: Trained noise-prediction network.
        n_samples: Number of points to generate.

    Returns:
        (n_samples, 2) generated data points.
    """
    model.eval()
    x = torch.randn(n_samples, 2, device=device)            # (n_samples, 2)

    for t_val in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)  # (n_samples,)
        eps_pred = model(x, t_batch)                         # (n_samples, 2)

        beta_t = betas[t_val]
        alpha_t = alphas[t_val]
        alpha_bar_t = alpha_bars[t_val]

        # DDPM mean
        x = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred
        )                                                    # (n_samples, 2)

        # Add noise for all steps except t=0
        if t_val > 0:
            z = torch.randn_like(x)                          # (n_samples, 2)
            x = x + torch.sqrt(beta_t) * z

    model.train()
    return x


samples = ddpm_sample(model)                                 # (2000, 2)

# ---------- Visualization ----------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(data[:, 0].cpu().numpy(), data[:, 1].cpu().numpy(), s=2, alpha=0.5)
axes[0].set_title("Training Data")
axes[0].set_aspect("equal")
axes[1].scatter(samples[:, 0].cpu().numpy(), samples[:, 1].cpu().numpy(), s=2, alpha=0.5, color="orange")
axes[1].set_title("Generated Samples (DDPM)")
axes[1].set_aspect("equal")
plt.tight_layout()
plt.show()

### Exercise 10.1 -- Debrief

**Common Mistakes:**
- Using `alpha_t` instead of `alpha_bar_t` in the forward process. Remember: `alpha_bar_t` is the *cumulative product* and controls total noise at step t.
- Forgetting `sqrt` on both the signal and noise coefficients. The forward process uses `sqrt(alpha_bar)` and `sqrt(1 - alpha_bar)`, not the raw values.
- Sampling noise at t=0 during the reverse process. The final step should be deterministic.
- Not normalizing the training data. Diffusion models assume data lives near the unit Gaussian scale.

**Talking Points for the Interviewer:**
- "The forward process is a closed-form Gaussian -- we can jump directly to any timestep t without iterating."
- "The reverse process mean has a `1/sqrt(alpha_t)` scaling and subtracts the predicted noise weighted by `beta_t / sqrt(1 - alpha_bar_t)`."
- "The variance in DDPM is fixed to `beta_t`; improved DDPM learns it."

---

## Exercise 10.2: DDIM Sampling with Classifier-Free Guidance (45 minutes)

**Set your timer to 45 minutes now.**

**Prompt:** Given a trained class-conditional diffusion model on MNIST, implement DDIM sampling with classifier-free guidance (CFG). Generate a grid of digits at multiple guidance scales.

**Evaluation Criteria:**
- Correct DDIM update rule with `predicted_x0` computation
- Proper CFG: one forward pass with class label, one with null label, linear combination
- Timestep sub-selection for accelerated sampling
- Clean grid visualization at guidance scales 1, 2, 4, 8

In [ ]:
# ---------- Compact class-conditional UNet for MNIST ----------
# This cell defines and trains the model so that Exercise 10.2 has a working checkpoint.

torch.manual_seed(42)

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# --- Data ---
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
loader = DataLoader(mnist, batch_size=128, shuffle=True, drop_last=True)

# --- Schedule (reuse T=300) ---
T_mnist = 300
betas_m = torch.linspace(1e-4, 0.02, T_mnist).to(device)       # (T,)
alphas_m = 1.0 - betas_m                                        # (T,)
alpha_bars_m = torch.cumprod(alphas_m, dim=0)                    # (T,)

NUM_CLASSES = 10
NULL_CLASS = NUM_CLASSES  # label index for unconditional (dropout)


class ResBlock(nn.Module):
    """Residual block with timestep and class conditioning."""

    def __init__(self, ch: int, emb_dim: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, ch)

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, ch, H, W)
            emb: (B, emb_dim) combined time+class embedding
        Returns:
            (B, ch, H, W)
        """
        h = F.silu(self.norm1(x))                                # (B, ch, H, W)
        h = self.conv1(h)                                        # (B, ch, H, W)
        h = h + self.emb_proj(emb)[:, :, None, None]             # broadcast (B, ch, 1, 1)
        h = F.silu(self.norm2(h))                                # (B, ch, H, W)
        h = self.conv2(h)                                        # (B, ch, H, W)
        return x + h                                             # residual


class SmallCondUNet(nn.Module):
    """Minimal class-conditional UNet for 28x28 MNIST images."""

    def __init__(self, in_ch: int = 1, base_ch: int = 32, emb_dim: int = 64):
        super().__init__()
        self.emb_dim = emb_dim
        # Embeddings
        self.time_mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 2), nn.SiLU(), nn.Linear(emb_dim * 2, emb_dim)
        )
        self.class_emb = nn.Embedding(NUM_CLASSES + 1, emb_dim)  # +1 for null class

        # Encoder
        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)          # 28x28
        self.down1 = nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1)  # 14x14
        self.rb1 = ResBlock(base_ch * 2, emb_dim)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1)  # 7x7
        self.rb2 = ResBlock(base_ch * 4, emb_dim)

        # Bottleneck
        self.mid = ResBlock(base_ch * 4, emb_dim)

        # Decoder
        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, stride=2, padding=1)  # 14x14
        self.rb3 = ResBlock(base_ch * 4, emb_dim)  # skip cat doubles channels
        self.up1 = nn.ConvTranspose2d(base_ch * 4, base_ch, 4, stride=2, padding=1)  # 28x28
        self.rb4 = ResBlock(base_ch * 2, emb_dim)

        self.out_conv = nn.Conv2d(base_ch * 2, in_ch, 3, padding=1)

    def _sinusoidal_emb(self, t: torch.Tensor) -> torch.Tensor:
        half = self.emb_dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None, :]             # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, emb_dim)

    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 1, 28, 28) noisy image in [-1, 1].
            t: (B,) integer timesteps.
            y: (B,) class labels (0-9) or NULL_CLASS for unconditional.
        Returns:
            (B, 1, 28, 28) predicted noise.
        """
        emb = self.time_mlp(self._sinusoidal_emb(t)) + self.class_emb(y)  # (B, emb_dim)

        h1 = self.in_conv(x)                    # (B, 32, 28, 28)
        h2 = self.rb1(self.down1(h1), emb)      # (B, 64, 14, 14)
        h3 = self.rb2(self.down2(h2), emb)      # (B, 128, 7, 7)
        h = self.mid(h3, emb)                   # (B, 128, 7, 7)
        h = self.up2(h)                         # (B, 64, 14, 14)
        h = self.rb3(torch.cat([h, h2], 1), emb)  # (B, 128, 14, 14) -> ResBlock(128) -> (B, 128, 14, 14)
        h = self.up1(h)                         # (B, 32, 28, 28)
        h = self.rb4(torch.cat([h, h1], 1), emb)  # (B, 64, 28, 28) -> ResBlock(64) -> (B, 64, 28, 28)
        return self.out_conv(h)                 # (B, 1, 28, 28)


cond_model = SmallCondUNet().to(device)
opt_m = torch.optim.Adam(cond_model.parameters(), lr=1e-3)
p_uncond = 0.1  # 10% chance of dropping class label for CFG training

step = 0
cond_model.train()
for epoch in range(2):  # ~2000 steps with batch 128
    for imgs, labels in tqdm(loader, desc=f"Epoch {epoch}"):
        imgs = imgs.to(device)                                   # (B, 1, 28, 28) in [-1,1]
        labels = labels.to(device)                               # (B,)

        # Random class dropout for CFG
        mask = torch.rand(len(labels), device=device) < p_uncond
        labels = torch.where(mask, torch.full_like(labels, NULL_CLASS), labels)  # (B,)

        t = torch.randint(0, T_mnist, (len(imgs),), device=device)  # (B,)
        eps = torch.randn_like(imgs)                             # (B, 1, 28, 28)
        ab = alpha_bars_m[t][:, None, None, None]                # (B, 1, 1, 1)
        x_t = torch.sqrt(ab) * imgs + torch.sqrt(1 - ab) * eps  # (B, 1, 28, 28)

        eps_pred = cond_model(x_t, t, labels)                    # (B, 1, 28, 28)
        loss = F.mse_loss(eps_pred, eps)

        opt_m.zero_grad()
        loss.backward()
        opt_m.step()
        step += 1
        if step >= 2000:
            break
    if step >= 2000:
        break

print(f"Trained {step} steps. Final loss: {loss.item():.4f}")

In [ ]:
# Starter code -- implement ddim_sample_cfg

def ddim_sample_cfg(
    model: nn.Module,
    num_steps: int = 50,
    guidance_scale: float = 2.0,
    class_label: int = 3,
    n_samples: int = 16,
) -> torch.Tensor:
    """Generate MNIST images using DDIM sampling with classifier-free guidance.

    Args:
        model: Trained class-conditional noise predictor.
        num_steps: Number of DDIM sub-steps (< T for acceleration).
        guidance_scale: CFG weight (w). eps_guided = eps_uncond + w * (eps_cond - eps_uncond).
        class_label: Target digit class (0-9).
        n_samples: Number of images to generate.

    Returns:
        (n_samples, 1, 28, 28) generated images in [-1, 1].
    """
    # YOUR CODE HERE
    pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Exercise 10.2: DDIM Sampling with CFG

@torch.no_grad()
def ddim_sample_cfg(
    model: nn.Module,
    num_steps: int = 50,
    guidance_scale: float = 2.0,
    class_label: int = 3,
    n_samples: int = 16,
) -> torch.Tensor:
    """Generate MNIST images using DDIM sampling with classifier-free guidance.

    Args:
        model: Trained class-conditional noise predictor.
        num_steps: Number of DDIM sub-steps (< T for acceleration).
        guidance_scale: CFG weight (w). eps_guided = eps_uncond + w * (eps_cond - eps_uncond).
        class_label: Target digit class (0-9).
        n_samples: Number of images to generate.

    Returns:
        (n_samples, 1, 28, 28) generated images clamped to [-1, 1].
    """
    model.eval()

    # Timestep sub-selection: uniform spacing from T-1 down to 0
    timesteps = torch.linspace(T_mnist - 1, 0, num_steps + 1).long().to(device)  # (num_steps+1,)

    x = torch.randn(n_samples, 1, 28, 28, device=device)        # (n, 1, 28, 28)

    for i in range(num_steps):
        t_cur = timesteps[i]       # current timestep
        t_next = timesteps[i + 1]  # next timestep (closer to 0)

        t_batch = t_cur.expand(n_samples)                        # (n,)

        # --- CFG: batch conditional + unconditional together ---
        x_double = torch.cat([x, x], dim=0)                     # (2n, 1, 28, 28)
        t_double = torch.cat([t_batch, t_batch], dim=0)          # (2n,)
        y_cond = torch.full((n_samples,), class_label, device=device, dtype=torch.long)
        y_uncond = torch.full((n_samples,), NULL_CLASS, device=device, dtype=torch.long)
        y_double = torch.cat([y_cond, y_uncond], dim=0)          # (2n,)

        eps_double = model(x_double, t_double, y_double)         # (2n, 1, 28, 28)
        eps_cond, eps_uncond = eps_double.chunk(2, dim=0)        # each (n, 1, 28, 28)

        # Guided noise prediction
        eps_guided = eps_uncond + guidance_scale * (eps_cond - eps_uncond)  # (n, 1, 28, 28)

        # --- DDIM update ---
        alpha_bar_t = alpha_bars_m[t_cur]
        alpha_bar_next = alpha_bars_m[t_next] if t_next > 0 else torch.tensor(1.0, device=device)

        # Predicted x_0
        predicted_x0 = (x - torch.sqrt(1 - alpha_bar_t) * eps_guided) / torch.sqrt(alpha_bar_t)
        predicted_x0 = predicted_x0.clamp(-1, 1)                # (n, 1, 28, 28)

        # Direction pointing to x_t
        direction = torch.sqrt(1 - alpha_bar_next) * eps_guided  # (n, 1, 28, 28)

        # DDIM deterministic step (eta=0)
        x = torch.sqrt(alpha_bar_next) * predicted_x0 + direction  # (n, 1, 28, 28)

    model.train()
    return x.clamp(-1, 1)


# --- Generate grids at different guidance scales ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, w in zip(axes, [1.0, 2.0, 4.0, 8.0]):
    torch.manual_seed(42)
    imgs = ddim_sample_cfg(cond_model, num_steps=50, guidance_scale=w, class_label=3, n_samples=16)
    # Make a 4x4 grid
    grid = imgs.cpu().view(4, 4, 28, 28)                        # (4, 4, 28, 28)
    grid = grid.permute(0, 2, 1, 3).reshape(4 * 28, 4 * 28)     # (112, 112)
    ax.imshow(grid.numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"w = {w}")
    ax.axis("off")

plt.suptitle("DDIM + CFG: Digit 3 at Various Guidance Scales", fontsize=14)
plt.tight_layout()
plt.show()

### Exercise 10.2 -- Debrief

**Key Implementation Details:**
- DDIM sub-selects timesteps (e.g. 50 out of 300) with uniform spacing. This allows fewer forward passes without retraining.
- The DDIM update is deterministic (eta=0): `x_{t-1} = sqrt(alpha_bar_{t-1}) * predicted_x0 + sqrt(1 - alpha_bar_{t-1}) * eps_predicted`.
- CFG batches the conditional and unconditional forward passes together for efficiency: `eps_guided = eps_uncond + w * (eps_cond - eps_uncond)`.
- Higher guidance scale increases sample fidelity to the class but reduces diversity. Very high w can cause artifacts.

**Common Mistakes:**
- Using `alpha_t` instead of `alpha_bar_t` in the DDIM formula.
- Forgetting to clamp `predicted_x0` -- this stabilizes sampling, especially at high guidance scales.
- Reversing the CFG formula direction (subtracting cond from uncond instead of the other way around).
- Not passing `NULL_CLASS` for the unconditional branch.

---

## Exercise 10.3: Training Step from Scratch (45 minutes)

**Set your timer to 45 minutes now.**

**Prompt:** You are given a pre-defined UNet model and MNIST data loader. Write the noise schedule, a `train_step` function that performs one diffusion training step, and a training loop of 500 steps. Plot the loss curve.

**Evaluation Criteria:**
- Correct noise schedule construction (betas -> alphas -> alpha_bars)
- `train_step` returns scalar loss and handles: sampling t, sampling noise, computing x_t, predicting noise, computing MSE, backprop
- Proper optimizer usage (zero_grad before backward, step after)
- Loss curve should show convergence

In [ ]:
# Starter code -- Exercise 10.3
# Model and data are already available from Exercise 10.2:
#   - SmallCondUNet class is defined
#   - MNIST DataLoader `loader` is ready
#   - device is set

# Create a fresh model for this exercise
torch.manual_seed(42)
model_ex3 = SmallCondUNet().to(device)
optimizer_ex3 = torch.optim.Adam(model_ex3.parameters(), lr=1e-3)

# YOUR CODE HERE:
# 1. Define noise schedule (T, betas, alphas, alpha_bars)
# 2. Write train_step(model, optimizer, x_batch, y_batch) -> float
# 3. Run 500 training steps, collect losses
# 4. Plot loss curve

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Exercise 10.3: Training Step from Scratch

torch.manual_seed(42)

# ---------- 1. Noise schedule ----------
T_ex3 = 300
betas_ex3 = torch.linspace(1e-4, 0.02, T_ex3).to(device)          # (T,)
alphas_ex3 = 1.0 - betas_ex3                                        # (T,)
alpha_bars_ex3 = torch.cumprod(alphas_ex3, dim=0)                    # (T,)

model_ex3 = SmallCondUNet().to(device)
optimizer_ex3 = torch.optim.Adam(model_ex3.parameters(), lr=1e-3)
p_uncond_ex3 = 0.1


# ---------- 2. train_step ----------
def train_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    x_batch: torch.Tensor,
    y_batch: torch.Tensor,
) -> float:
    """Perform one diffusion training step.

    Args:
        model: Noise prediction network (class-conditional).
        optimizer: Optimizer for model parameters.
        x_batch: (B, 1, 28, 28) clean images in [-1, 1].
        y_batch: (B,) class labels.

    Returns:
        Scalar MSE loss value.
    """
    B = x_batch.shape[0]

    # Class dropout for CFG
    mask = torch.rand(B, device=device) < p_uncond_ex3
    y_batch = torch.where(mask, torch.full_like(y_batch, NULL_CLASS), y_batch)  # (B,)

    # Sample random timesteps
    t = torch.randint(0, T_ex3, (B,), device=device)                # (B,)

    # Sample noise
    eps = torch.randn_like(x_batch)                                  # (B, 1, 28, 28)

    # Compute x_t via forward process
    ab = alpha_bars_ex3[t][:, None, None, None]                      # (B, 1, 1, 1)
    x_t = torch.sqrt(ab) * x_batch + torch.sqrt(1 - ab) * eps       # (B, 1, 28, 28)

    # Predict noise
    eps_pred = model(x_t, t, y_batch)                                # (B, 1, 28, 28)

    # MSE loss
    loss = F.mse_loss(eps_pred, eps)

    # Backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


# ---------- 3. Training loop ----------
model_ex3.train()
losses_ex3 = []
data_iter = iter(loader)
for step in tqdm(range(500), desc="Training (Ex 10.3)"):
    try:
        imgs, labels = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        imgs, labels = next(data_iter)

    imgs = imgs.to(device)      # (B, 1, 28, 28)
    labels = labels.to(device)  # (B,)

    loss_val = train_step(model_ex3, optimizer_ex3, imgs, labels)
    losses_ex3.append(loss_val)

# ---------- 4. Plot loss curve ----------
plt.figure(figsize=(8, 3))
plt.plot(losses_ex3, alpha=0.4, label="Raw loss")
window = 50
smoothed = np.convolve(losses_ex3, np.ones(window) / window, mode="valid")
plt.plot(range(window - 1, len(losses_ex3)), smoothed, color="red", label=f"Smoothed ({window}-step)")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Exercise 10.3: Training Loss Curve (500 steps)")
plt.legend()
plt.show()

### Exercise 10.3 -- Debrief

**What the Interviewer Is Looking For:**
- Can you decompose diffusion training into its atomic steps from memory?
- Do you understand the roles of each schedule tensor (betas, alphas, alpha_bars)?
- Is your `train_step` clean and modular (could drop it into a larger codebase)?

**Common Mistakes:**
- Calling `loss.backward()` before `optimizer.zero_grad()` -- gradients accumulate from the previous step.
- Forgetting to convert alpha_bars to shape `(B, 1, 1, 1)` for broadcasting with images.
- Using `t` as a float directly instead of indexing into `alpha_bars` with integer timesteps.
- Not including class label dropout during training (required for CFG at inference time).

---

## Exercise 10.4: Verbal Explanation Prompts

For each question below, **speak your answer aloud** (or write it down) before expanding the model answer. In a real interview, you will need to explain these concepts clearly and concisely without code in front of you.

---

### Core Concepts

#### Q1: Walk me through the forward diffusion process.

*Try to answer before reading the model answer below.*

**A1: Model Answer**

The forward process gradually adds Gaussian noise to data over T timesteps. At each step t, we add a small amount of noise controlled by a variance schedule $\beta_t$:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t I)$$

The key insight is that we can skip directly to any timestep t using the closed-form:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar\alpha_t}\, x_0,\; (1-\bar\alpha_t) I)$$

where $\bar\alpha_t = \prod_{s=1}^t (1 - \beta_s)$. This means: $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon$, with $\epsilon \sim \mathcal{N}(0, I)$.

At $t=0$, we have clean data. At $t=T$, we have approximately pure Gaussian noise.

**Key Points:** Closed-form jump to any t. Signal attenuates by $\sqrt{\bar\alpha_t}$, noise scales by $\sqrt{1 - \bar\alpha_t}$. These coefficients ensure the variance is preserved (SNR decreases monotonically).

**Common Mistake:** Confusing $\alpha_t$ (single-step) with $\bar\alpha_t$ (cumulative product). The forward process to arbitrary t uses $\bar\alpha_t$.

#### Q2: Why do we predict noise instead of the clean image?

*Try to answer before reading the model answer below.*

**A2: Model Answer**

Predicting noise ($\epsilon$-prediction) works better empirically than predicting $x_0$ directly, for several reasons:

1. **Uniform difficulty across timesteps.** The noise $\epsilon$ is always drawn from $\mathcal{N}(0, I)$ regardless of $t$. If we predicted $x_0$ directly, the task would be trivially easy at low $t$ (barely noisy) and impossibly hard at high $t$ (nearly pure noise), creating an imbalanced loss landscape.

2. **Connection to score matching.** Predicting $\epsilon$ is equivalent to estimating the score function $\nabla_{x_t} \log q(x_t)$, which has deep theoretical roots in score-based generative modeling. Specifically, $\epsilon_\theta(x_t, t) \propto -\nabla_{x_t} \log q(x_t)$.

3. **Simplified loss.** The DDPM simplified loss $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$ drops the timestep-dependent weighting from the variational bound, and this simpler objective trains more stably.

**Key Point:** All three parameterizations ($\epsilon$, $x_0$, $v$) are mathematically interconvertible. The choice affects training dynamics, not expressiveness.

**Common Mistake:** Claiming noise prediction is "theoretically better." It is empirically better for standard diffusion; $x_0$-prediction can be superior in other settings (e.g., discrete diffusion).

#### Q3: Explain the reparameterization trick and why it matters for diffusion.

*Try to answer before reading the model answer below.*

**A3: Model Answer**

The reparameterization trick rewrites a sample from a parameterized distribution as a deterministic function of the parameters plus independent noise. Instead of sampling $x \sim \mathcal{N}(\mu, \sigma^2)$ directly (which is not differentiable w.r.t. $\mu, \sigma$), we write:

$$x = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, 1)$$

Now gradients flow through $\mu$ and $\sigma$ because the randomness is isolated in $\epsilon$.

In diffusion, this trick is used in two places:
1. **Forward process:** $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon$ -- this is literally the reparameterized form of $q(x_t|x_0)$.
2. **Training:** We sample $\epsilon$ independently, construct $x_t$ deterministically from $(x_0, \epsilon, t)$, and compute loss. Gradients flow cleanly back through the model.

**Key Point:** Without reparameterization, we could not backpropagate through the stochastic sampling step. This trick (originally from VAEs) is what makes the entire training pipeline differentiable.

**Common Mistake:** Thinking the reparameterization trick is specific to diffusion. It is a general technique used in VAEs, normalizing flows, and any setting where you need gradients through a sampling operation.

#### Q4: Derive or explain the simplified DDPM loss.

*Try to answer before reading the model answer below.*

**A4: Model Answer**

The full DDPM objective comes from the variational lower bound (VLB) on $\log p(x_0)$. It decomposes into T KL-divergence terms, one per timestep. Each term compares the true posterior $q(x_{t-1}|x_t, x_0)$ to the learned reverse $p_\theta(x_{t-1}|x_t)$.

When both are Gaussian with fixed variance, the KL simplifies to an MSE between means. Substituting the $\epsilon$-parameterization and rearranging, each term becomes:

$$L_t = \frac{\beta_t^2}{2\sigma_t^2 \alpha_t (1 - \bar\alpha_t)} \|\epsilon - \epsilon_\theta(x_t, t)\|^2$$

The **simplified loss** drops the timestep-dependent weighting coefficient:

$$L_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

Ho et al. (2020) found that this unweighted version trains better in practice. It upweights large-$t$ terms (which have small coefficients in the VLB), encouraging the model to denoise from highly corrupted inputs.

**Key Point:** The simplified loss is not the true VLB -- it is a reweighted version that empirically produces better samples at the cost of a looser bound.

**Common Mistake:** Claiming the simplified loss IS the variational bound. It is a simplification that discards per-timestep weighting.

#### Q5: Compare DDPM and DDIM sampling.

*Try to answer before reading the model answer below.*

**A5: Model Answer**

| Property | DDPM | DDIM |
|---|---|---|
| Stochasticity | Stochastic (adds noise at each step) | Deterministic (eta=0) or controllable |
| Steps required | All T steps (e.g. 1000) | Any subset (e.g. 50 steps) |
| Same model? | Yes | Yes -- no retraining needed |
| Latent interpolation | Not meaningful (stochastic) | Meaningful (deterministic mapping) |
| Sample quality at few steps | Degrades significantly | Maintains quality |

**How DDIM works:** Instead of the Markov reverse chain, DDIM defines a non-Markovian process with the same marginals $q(x_t|x_0)$. The update rule first computes `predicted_x0` from the current `x_t` and predicted noise, then deterministically interpolates toward the next (less noisy) timestep:

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \cdot \hat{x}_0 + \sqrt{1-\bar\alpha_{t-1}} \cdot \epsilon_\theta(x_t, t)$$

The `eta` parameter controls how much stochastic noise to inject. At eta=0, sampling is fully deterministic; at eta=1, it recovers DDPM.

**Key Point:** DDIM can use the same trained model as DDPM. The only difference is the sampling procedure.

**Common Mistake:** Thinking DDIM requires a different training procedure. It does not -- only the inference loop changes.

---

### Architecture

#### Q6: Why use GroupNorm instead of BatchNorm in diffusion UNets?

*Try to answer before reading the model answer below.*

**A6: Model Answer**

GroupNorm normalizes within groups of channels for each sample independently, while BatchNorm normalizes across the batch dimension.

GroupNorm is preferred for diffusion models because:

1. **Batch independence.** During sampling, we often generate a single image (batch size 1). BatchNorm statistics become meaningless with B=1. GroupNorm works identically regardless of batch size.

2. **Varying noise levels in a batch.** In diffusion training, each sample in the batch has a different noise level $t$. BatchNorm would compute statistics across samples at wildly different noise levels, mixing signals inappropriately. GroupNorm computes per-sample statistics, avoiding this issue.

3. **Stable training.** GroupNorm avoids the train/eval discrepancy of BatchNorm's running statistics, which is especially problematic for the long training runs typical of diffusion models.

**Key Point:** Any normalization that depends on the batch dimension is problematic when batch elements have heterogeneous conditioning (different timesteps).

**Common Mistake:** Saying "GroupNorm is always better than BatchNorm." BatchNorm is fine when the batch is homogeneous and large (e.g., classification). The issue is specific to diffusion's per-sample timestep conditioning.

#### Q7: Where should attention layers be placed in a diffusion UNet, and why?

*Try to answer before reading the model answer below.*

**A7: Model Answer**

Attention layers are typically placed at **lower spatial resolutions** in the UNet (e.g., 16x16 or 8x8), not at full resolution. This is a compute/quality tradeoff:

1. **Quadratic cost.** Self-attention has $O(n^2)$ complexity where $n$ is the number of spatial tokens ($H \times W$). At 64x64, that is 4096 tokens -- expensive but feasible. At 256x256 (65536 tokens), it would be prohibitive.

2. **Global context where it matters.** At low resolutions, each spatial position already has a large receptive field from the preceding convolutions. Attention at these levels captures long-range semantic relationships (e.g., symmetry, global structure) that convolutions alone miss.

3. **Cross-attention for conditioning.** In text-to-image models (Stable Diffusion), cross-attention layers at these resolutions attend to text embeddings, enabling spatial control of where concepts appear.

**Typical placement:** After ResBlocks at 16x16 and 8x8 resolutions. Some architectures (e.g., DiT) replace convolutions entirely with transformers, applying attention at all levels using patch embeddings.

**Common Mistake:** Placing attention at full resolution. This explodes memory and compute without proportional quality gains -- convolutions already handle local patterns effectively.

#### Q8: How is the timestep conditioned into the network?

*Try to answer before reading the model answer below.*

**A8: Model Answer**

Timestep conditioning follows a two-stage process:

**Stage 1: Embedding.** The integer timestep $t$ is mapped to a high-dimensional vector using sinusoidal positional encoding (borrowed from transformers):
$$\text{emb}(t) = [\sin(t \cdot \omega_1), \cos(t \cdot \omega_1), \ldots, \sin(t \cdot \omega_d), \cos(t \cdot \omega_d)]$$
where $\omega_i$ are geometrically spaced frequencies. This is then passed through a small MLP (2 linear layers with SiLU) to produce a learned embedding.

**Stage 2: Injection into residual blocks.** The embedding vector is projected to match the channel dimension and added (or used for scale+shift via AdaGN) to the feature maps inside each ResBlock:
- **Additive:** `h = h + Linear(emb)[:, :, None, None]` -- simple broadcast addition.
- **AdaGN (Adaptive Group Norm):** The embedding produces scale ($\gamma$) and shift ($\beta$) parameters that modulate the normalized features: `h = gamma * GroupNorm(h) + beta`. This is more expressive and used in DiT and modern architectures.

**Key Point:** The timestep must be injected at every resolution level of the UNet, not just once. The network needs to know the noise level throughout its computation.

**Common Mistake:** Feeding raw integer $t$ directly as a scalar input. Without sinusoidal embedding, the network struggles to distinguish between nearby timesteps and cannot generalize across the continuous noise level.

#### Q9: Why are skip connections essential in diffusion UNets?

*Try to answer before reading the model answer below.*

**A9: Model Answer**

Skip connections concatenate (or add) encoder features to the corresponding decoder features at the same spatial resolution.

They are essential for diffusion because:

1. **Preserving spatial detail.** The encoder downsamples to capture global context, but fine-grained spatial information (edges, textures) is lost. Skip connections provide a direct path for high-resolution details to reach the decoder, enabling pixel-precise noise prediction.

2. **The denoising task is a residual problem.** At low noise levels, $x_t \approx x_0$ and the predicted noise is small. Skip connections let the network effectively learn a residual -- the decoder can "copy" the input and add small corrections, rather than reconstructing everything from the bottleneck.

3. **Gradient flow.** Skip connections provide short paths for gradients during backpropagation, mitigating vanishing gradients in deep networks. This is the same benefit as in ResNets.

4. **Multi-scale features.** The encoder features at each level capture patterns at different scales. By concatenating them with decoder features, the network can combine both local (encoder) and global (bottleneck) information at each resolution.

**Key Point:** Without skip connections, the UNet becomes an autoencoder bottleneck, and the model cannot predict fine-grained noise patterns.

**Common Mistake:** Confusing UNet skip connections (cross-level concatenation) with ResBlock skip connections (within-block addition). Both exist in diffusion UNets and serve complementary purposes.

---

### Guidance

#### Q10: Explain classifier-free guidance (CFG).

*Try to answer before reading the model answer below.*

**A10: Model Answer**

Classifier-free guidance (CFG) is a technique that improves conditional sample quality without requiring a separate classifier. It works by training a single model to handle both conditional and unconditional generation, then amplifying the difference at inference time.

**Training:** With probability $p$ (typically 10-20%), the conditioning signal (class label, text embedding) is replaced with a null token. This teaches the model to generate both conditionally and unconditionally.

**Inference:** At each sampling step, run two forward passes:
- Conditional: $\epsilon_\theta(x_t, t, c)$ with the desired condition
- Unconditional: $\epsilon_\theta(x_t, t, \varnothing)$ with the null token

Combine them: $\tilde{\epsilon} = \epsilon_\text{uncond} + w \cdot (\epsilon_\text{cond} - \epsilon_\text{uncond})$

where $w$ is the guidance scale. This extrapolates in the direction of the conditioning signal.

**Intuition:** The difference $(\epsilon_\text{cond} - \epsilon_\text{uncond})$ represents "what the condition adds." Scaling it by $w > 1$ amplifies this effect, producing samples that are more strongly aligned with the condition.

**Key Point:** CFG replaced classifier guidance (which required a separately trained classifier on noisy images) and became the standard approach in all modern diffusion systems.

**Common Mistake:** Setting $w = 1$ and expecting CFG to have an effect. At $w = 1$, the formula reduces to plain conditional sampling with no guidance boost.

#### Q11: What happens as you increase the guidance scale? What are the tradeoffs?

*Try to answer before reading the model answer below.*

**A11: Model Answer**

| Guidance Scale (w) | Fidelity to Condition | Sample Diversity | Artifacts |
|---|---|---|---|
| w = 1 | Baseline (no boost) | High | None |
| w = 2-4 | Good | Moderate | Minimal |
| w = 7-10 | High (typical default) | Low | Some saturation |
| w > 15 | Very high | Very low | Oversaturation, color distortion |

**The core tradeoff is fidelity vs. diversity** (analogous to the precision/recall tradeoff in GANs, or the temperature parameter in language models):

- **Low w:** Samples are diverse but may not strongly match the condition. The model explores more of the learned distribution.
- **High w:** Samples are sharp and closely match the condition but collapse toward "prototypical" examples. Extreme values push predictions outside the training distribution, causing artifacts (saturated colors, distorted features).

**Why artifacts occur:** At very high $w$, the guided noise prediction $\tilde{\epsilon}$ can have magnitudes far beyond what the model was trained on, pushing $x_t$ into out-of-distribution regions at each sampling step.

**Practical mitigation:** Dynamic thresholding (Imagen) clips the predicted $x_0$ at each step to prevent extreme values. Rescaled CFG divides by the norm of the guided prediction.

**Common Mistake:** Thinking higher guidance is always better. There is a sweet spot (typically 7-12 for text-to-image) beyond which quality degrades.

#### Q12: How do negative prompts work in diffusion models?

*Try to answer before reading the model answer below.*

**A12: Model Answer**

Negative prompts are a simple modification of the CFG formula. Instead of using the null/empty embedding for the unconditional branch, you substitute a text embedding of what you want to *avoid*:

$$\tilde{\epsilon} = \epsilon_\theta(x_t, t, c_\text{neg}) + w \cdot (\epsilon_\theta(x_t, t, c_\text{pos}) - \epsilon_\theta(x_t, t, c_\text{neg}))$$

**How it works:** The guidance direction is now "move away from the negative prompt, toward the positive prompt." The model steers sampling away from the semantic region described by the negative prompt.

**Examples:**
- Positive: "a photo of a cat", Negative: "blurry, low quality" -- produces sharper images
- Positive: "landscape painting", Negative: "people, text, watermark" -- removes unwanted elements

**Why this works:** In the CFG framework, the unconditional branch serves as the "default direction." By replacing it with an undesirable direction, we define the guidance vector to explicitly point away from undesirable features.

**Key Point:** Negative prompts require no architectural changes or retraining. They are purely an inference-time modification of the CFG formula.

**Common Mistake:** Thinking negative prompts are handled by a separate model or loss function. They are just a swap of the unconditional embedding in the existing CFG computation.

---

### Advanced Topics

#### Q13: Compare pixel-space diffusion vs. latent diffusion.

*Try to answer before reading the model answer below.*

**A13: Model Answer**

| Property | Pixel-Space Diffusion | Latent Diffusion (LDM) |
|---|---|---|
| Operates on | Raw pixels (e.g. 512x512x3) | Compressed latent (e.g. 64x64x4) |
| Compute per step | Very high (large spatial dims) | Much lower (8x spatial compression) |
| Separate encoder needed? | No | Yes (pretrained VAE encoder + decoder) |
| Training cost | Extremely high | Moderate (diffusion only on latents) |
| Reconstruction quality | Pixel-perfect (no compression) | Slight VAE reconstruction loss |
| Examples | DDPM, Imagen (partially) | Stable Diffusion, SDXL, DALL-E 3 |

**How latent diffusion works:**
1. A VAE is pretrained to compress images: $z = \text{Encoder}(x)$, $\hat{x} = \text{Decoder}(z)$.
2. The diffusion model operates entirely in the latent space $z$, which is typically 48-64x smaller than pixel space.
3. At inference: sample $z_0$ via diffusion, then decode to pixels with the VAE decoder.

**Why latent diffusion won:** The computational savings are massive. A 512x512 image becomes a 64x64 latent, reducing memory and compute by ~64x per diffusion step. The VAE handles perceptually unimportant pixel-level details.

**Key Point:** The VAE is frozen during diffusion training. The two-stage approach decouples perceptual compression from semantic generation.

**Common Mistake:** Thinking latent diffusion is lossless. The VAE introduces reconstruction error, which bounds the maximum quality of the generated images.

#### Q14: What are the tradeoffs between epsilon-prediction, x0-prediction, and v-prediction?

*Try to answer before reading the model answer below.*

**A14: Model Answer**

All three parameterizations are mathematically equivalent -- you can convert between them given $x_t$ and $\bar\alpha_t$. They differ in training dynamics and practical behavior:

| Parameterization | Predicts | Loss weighting effect | Best for |
|---|---|---|---|
| $\epsilon$-prediction | Noise $\epsilon$ | Upweights high-noise timesteps | Standard DDPM, general purpose |
| $x_0$-prediction | Clean data $x_0$ | Upweights low-noise timesteps | Fast sampling, consistency models |
| $v$-prediction | $v = \sqrt{\bar\alpha_t}\epsilon - \sqrt{1-\bar\alpha_t}x_0$ | Balanced across timesteps | Progressive distillation, SD 2.x |

**Conversion formulas (given $x_t$, $\bar\alpha_t$):**
- From $\epsilon$: $\hat{x}_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\epsilon) / \sqrt{\bar\alpha_t}$
- From $x_0$: $\hat{\epsilon} = (x_t - \sqrt{\bar\alpha_t}\,x_0) / \sqrt{1-\bar\alpha_t}$
- $v$-prediction interpolates: $v$ rotates between $\epsilon$ and $x_0$ predictions based on the SNR

**Why v-prediction emerged:** At very high noise ($t \approx T$), $\epsilon$-prediction works well but $x_0$-prediction fails (nearly impossible to reconstruct from pure noise). At very low noise ($t \approx 0$), the reverse happens. $v$-prediction provides a smooth interpolation that works well across all timesteps.

**Key Point:** The choice of parameterization implicitly changes the loss weighting across timesteps, even without explicitly modifying the loss function.

**Common Mistake:** Treating these as fundamentally different models. They are the same model with different output interpretations and equivalent at convergence (in theory).

#### Q15: How does flow matching differ from standard diffusion?

*Try to answer before reading the model answer below.*

**A15: Model Answer**

Flow matching defines a straight-line interpolation between noise and data, rather than the curved noising process of diffusion:

| Property | Diffusion (DDPM/DDIM) | Flow Matching |
|---|---|---|
| Forward path | Curved (geometric mean of signal/noise) | Straight line: $x_t = (1-t)x_0 + t\epsilon$ |
| Model predicts | Noise $\epsilon$ or score | Velocity field $v_t = \frac{dx_t}{dt}$ |
| Time range | Discrete $t \in \{0, \ldots, T\}$ | Continuous $t \in [0, 1]$ |
| Sampling | DDPM/DDIM update rules | ODE integration (Euler, Heun, etc.) |
| Schedule design | Critical (beta schedule) | Not needed (straight paths) |
| Training target | $v_t = \epsilon - x_0$ (the conditional flow) | Same formula, cleaner derivation |

**How flow matching works:**
1. Define an interpolation: $x_t = (1-t) x_0 + t \epsilon$, where $\epsilon \sim \mathcal{N}(0,I)$.
2. The velocity along this path is $v_t = \epsilon - x_0$ (constant, independent of $t$).
3. Train a network $v_\theta(x_t, t)$ to predict this velocity via MSE: $\|v_\theta(x_t, t) - (\epsilon - x_0)\|^2$.
4. Sample by integrating the ODE from $t=1$ (noise) to $t=0$ (data): $dx = v_\theta(x, t) dt$.

**Advantages of flow matching:** Simpler derivation (no VLB needed), straighter paths require fewer ODE steps for the same quality, and the framework naturally extends to non-Gaussian source distributions.

**Key Point:** Flow matching and diffusion converge to similar practical implementations. The main difference is conceptual clarity and the straightness of the sampling trajectory.

**Common Mistake:** Saying flow matching is "not diffusion." It is a different formulation of the same generative modeling paradigm, and the trained models are often interchangeable.

#### Q16: What challenges arise when scaling diffusion to 512x512 images, and how are they addressed?

*Try to answer before reading the model answer below.*

**A16: Model Answer**

Scaling from 28x28 MNIST to 512x512 natural images introduces several orders of magnitude more complexity:

**1. Compute and Memory:**
- A 512x512x3 image has ~786K pixels vs. 784 for MNIST (1000x larger).
- Solution: **Latent diffusion** -- compress to 64x64x4 latents via a pretrained VAE, reducing the problem by ~64x.

**2. Architecture:**
- Need much larger UNets (hundreds of millions of parameters). ADM uses ~550M params.
- Attention at 64x64 is expensive (4K tokens). Solution: attention only at 32x32 and 16x16 resolutions.
- Modern alternative: **DiT (Diffusion Transformer)** -- uses patchified transformer instead of UNet, scales better with compute.

**3. Training:**
- Requires millions of image-text pairs and hundreds of GPU-days.
- Mixed precision (fp16/bf16) is essential for memory and speed.
- EMA (exponential moving average) of model weights for stable sampling.
- Gradient accumulation to simulate large effective batch sizes.

**4. Sampling Speed:**
- 1000-step DDPM at 512x512 is impractically slow.
- Solutions: DDIM (50 steps), DPM-Solver (20 steps), distillation (1-4 steps), consistency models (1 step).

**5. Conditioning:**
- Text conditioning via cross-attention with CLIP or T5 text embeddings.
- Micro-conditioning on resolution and crop parameters (SDXL).

**Key Point:** The algorithmic framework is identical to MNIST diffusion. The differences are engineering: compression, architecture scale, training infrastructure, and sampling acceleration.

---

## Exercise 10.5: Bug Finding Challenge

The code below implements diffusion training and sampling but contains **10 hidden bugs**. Each bug is plausible (the code runs or nearly runs) but produces incorrect results.

Read through the code carefully and identify all 10 bugs. For each bug, explain what is wrong and how to fix it. Write your answers before checking the solution.

In [ ]:
# BUGGY CODE -- Find all 10 bugs!
# This code is intentionally broken. Do NOT run this cell.
# Read it carefully and identify the bugs.

buggy_code = """
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = alphas

class BuggyDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + 1, 256),
            nn.ReLU(),
            nn.Linear(256, 2),
        )

    def forward(self, x, t):
      
        t_input = t.unsqueeze(-1).float()  # just a scalar
        return self.net(torch.cat([x, t_input], dim=-1))

# --- Training ---
model = BuggyDenoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
data = torch.randn(1000, 2)  # assume some training data

for step in range(1000):
    idx = torch.randint(0, len(data), (64,))
    x_0 = data[idx]

    t = torch.randint(0, T, (64,))
    eps = torch.randn_like(x_0)

    # --- Bug 3: Missing sqrt ---
    ab_t = alpha_bars[t].unsqueeze(-1)
    x_t = ab_t * x_0 + (1 - ab_t) * eps

    eps_pred = model(x_t, t)

    loss = F.mse_loss(eps_pred, eps)

    # --- Bug 4: zero_grad after backward ---
    loss.backward()
    optimizer.zero_grad()
    optimizer.step()

# Images loaded as [0, 1] but not normalized to [-1, 1]
# transform = transforms.ToTensor()

# --- Sampling ---
def sample(model, n_samples=100):
    x = torch.randn(n_samples, 2)

    # --- Bug 6: Off-by-one, sampling includes t=0 noise addition ---
    for t_val in reversed(range(T)):  # goes T-1 ... 0
        t_batch = torch.full((n_samples,), t_val, dtype=torch.long)
        eps_pred = model(x, t_batch)

        beta_t = betas[t_val]
        alpha_t = alphas[t_val]

        # --- Bug 7: Using alpha_t instead of alpha_bar_t ---
        x = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alpha_t)) * eps_pred
        )

        # --- Bug 8: Adding noise at t=0 ---
        z = torch.randn_like(x)
        x = x + torch.sqrt(beta_t) * z

    return x

# model.eval() not called before sampling -- dropout/batchnorm behave differently

# eps_guided = eps_cond + w * (eps_cond - eps_uncond)
#   or equivalently: (1 - w) * eps_uncond + w * eps_cond
"""

print("Read the code above and find all 10 bugs before checking the solution below.")

# ✅ SOLUTION — try the exercise above before running this -- Exercise 10.5: All 10 Bugs Explained

### Bug 1: `alpha_bars = alphas` (no cumulative product)
**What is wrong:** `alpha_bars` should be the cumulative product of `alphas`, not just a copy. Without `cumprod`, each `alpha_bar_t` equals a single-step `alpha_t` instead of the product up to step $t$.
**Fix:** `alpha_bars = torch.cumprod(alphas, dim=0)`

### Bug 2: Raw timestep instead of sinusoidal embedding
**What is wrong:** Feeding the raw integer $t$ as a single scalar gives the network almost no ability to distinguish between nearby timesteps (e.g., t=500 vs t=501). Sinusoidal embeddings provide a rich, high-dimensional representation.
**Fix:** Use a `SinusoidalEmbedding` module that maps $t$ to a vector of dimension 64+, then concatenate with $x$.

### Bug 3: Missing `sqrt` in forward process
**What is wrong:** The forward process should be $x_t = \sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t} \epsilon$. The code uses $\bar\alpha_t$ and $(1-\bar\alpha_t)$ without square roots, which changes the signal-to-noise ratio and breaks the variance-preserving property.
**Fix:** `x_t = torch.sqrt(ab_t) * x_0 + torch.sqrt(1 - ab_t) * eps`

### Bug 4: `optimizer.zero_grad()` called AFTER `loss.backward()`
**What is wrong:** `zero_grad()` clears gradients. Calling it after `backward()` erases the gradients that were just computed, so `optimizer.step()` updates with zero gradients. The model never learns.
**Fix:** Call `optimizer.zero_grad()` BEFORE `loss.backward()`.

### Bug 5: Images in [0, 1] instead of [-1, 1]
**What is wrong:** Diffusion assumes data centered near zero. Images in [0, 1] have mean 0.5, so at $t=T$ the distribution is $\mathcal{N}(0.5 \cdot \sqrt{\bar\alpha_T}, 1-\bar\alpha_T)$ instead of $\mathcal{N}(0, 1)$. Sampling starts from $\mathcal{N}(0, 1)$, creating a distribution mismatch.
**Fix:** Normalize to [-1, 1]: `transforms.Normalize((0.5,), (0.5,))` after `ToTensor()`.

### Bug 6: Off-by-one in sampling loop (not a separate bug here, but combined with Bug 8)
**What is wrong:** The loop `reversed(range(T))` goes from $T-1$ down to $0$, which is correct for the iteration range. However, the noise addition at $t=0$ (Bug 8) makes the final output noisy.
**Note:** The loop bounds themselves are correct; the real issue is adding noise at $t=0$.

### Bug 7: Using `alpha_t` instead of `alpha_bar_t` in reverse mean
**What is wrong:** The DDPM reverse mean formula uses $\sqrt{1 - \bar\alpha_t}$ in the denominator, not $\sqrt{1 - \alpha_t}$. Using the single-step $\alpha_t$ dramatically changes the noise coefficient.
**Fix:** `x = (1/sqrt(alpha_t)) * (x - (beta_t / sqrt(1 - alpha_bars[t_val])) * eps_pred)`

### Bug 8: Adding noise at t=0
**What is wrong:** The final denoising step ($t=0$) should be deterministic -- no noise added. Adding noise at $t=0$ corrupts the final clean sample.
**Fix:** `if t_val > 0: x = x + torch.sqrt(beta_t) * z`

### Bug 9: Missing `model.eval()` during sampling
**What is wrong:** Without `model.eval()`, dropout layers remain active (randomly zeroing activations) and BatchNorm uses batch statistics instead of running statistics. This produces inconsistent, degraded samples.
**Fix:** Call `model.eval()` before sampling, `model.train()` after.

### Bug 10: Wrong CFG formula
**What is wrong:** The buggy formula `eps_cond + w * (eps_cond - eps_uncond)` is equivalent to `(1+w) * eps_cond - w * eps_uncond`, which over-amplifies the conditional signal.
**Fix:** `eps_guided = eps_uncond + w * (eps_cond - eps_uncond)`. At $w=1$ this gives plain conditional; at $w>1$ it extrapolates.

---

## Exercise 10.6: Design Discussion

**Prompt:** You have trained a diffusion model on 28x28 MNIST. Your manager asks you to scale it to generate 512x512 photorealistic images. Walk through the major design decisions, architecture changes, and engineering considerations. You have a team of 3 engineers and 8 A100 GPUs.

*Organize your answer into clear sections. Think about this for 10 minutes before reading the model answer.*

### Model Answer: Scaling Diffusion to 512x512

**1. Latent Diffusion (First Priority)**

Operating in pixel space at 512x512 is impractical. Use a two-stage approach:
- Train or use a pretrained VAE (e.g., from Stable Diffusion) that compresses 512x512x3 to 64x64x4 (8x spatial downsampling).
- Run diffusion entirely in this latent space. This reduces compute per diffusion step by ~64x.
- The VAE is trained separately and frozen during diffusion training.

**2. Architecture Changes**

- Scale the UNet from ~1M to ~200-500M parameters. Channel multipliers: [1, 2, 4, 4] at resolutions [64, 32, 16, 8].
- Add self-attention at 32x32 and 16x16 resolutions (not at 64x64 -- too expensive).
- Add cross-attention for text conditioning (CLIP or T5 text encoder).
- Use GroupNorm throughout (not BatchNorm).
- Consider DiT (Diffusion Transformer) as an alternative if you have sufficient compute.

**3. Training Infrastructure**

- **Data:** Need millions of high-quality image-text pairs. Curate carefully (NSFW filtering, deduplication, quality filtering).
- **Distributed training:** Data-parallel across 8 A100s with gradient accumulation. Effective batch size ~256-2048.
- **Mixed precision:** bf16 for forward/backward, fp32 for optimizer states.
- **EMA:** Maintain an exponential moving average of model weights (decay 0.9999) for sampling.
- **Training time:** Expect weeks to months even on 8 A100s.

**4. Noise Schedule**

- Linear schedule may not be optimal at this scale. Consider cosine schedule (improved DDPM) or learned schedule.
- More timesteps may be needed (T=1000 is standard).

**5. Inference Optimization**

- DDIM sampling with 50 steps for quick iteration during development.
- DPM-Solver++ (20 steps) or distilled models for production.
- Classifier-free guidance with w=7.5 (typical default).
- Half-precision inference to double throughput.

**6. Evaluation**

- FID (Frechet Inception Distance) on a standard benchmark (e.g., COCO-30K).
- CLIP score for text-image alignment.
- Human evaluation for subjective quality.

---

**Follow-up Questions an Interviewer Might Ask:**
- "How would you handle variable aspect ratios?" (Bucket by aspect ratio, pad within buckets, use positional encoding that generalizes.)
- "How would you add ControlNet-style conditioning?" (Freeze the base model, train a parallel encoder that injects features via zero-initialized convolutions.)
- "How would you reduce inference latency to under 1 second?" (Distillation to 4 steps, or consistency models for 1-step generation. Quantize to INT8.)
- "What if you only have 2 GPUs?" (Use a smaller model, LoRA fine-tuning on a pretrained checkpoint, or fine-tune only the attention layers.)

---

## Final Section: Interview Tips

### 7 Tips for Diffusion Model Interviews

**1. Think aloud.** Narrate your thought process as you write code. The interviewer cares as much about how you think as what you produce. Say things like "First I need the noise schedule, then the forward process, then the model, then training." This shows structured thinking even if you make a mistake.

**2. Start simple, then refine.** Begin with the simplest version that works (e.g., MLP denoiser, 2D data) before adding complexity. Interviewers appreciate incremental development over ambitious but incomplete solutions. A working 2D spiral diffusion is better than a half-finished UNet.

**3. Write clean code.** Use descriptive variable names (`alpha_bars` not `ab`), add shape comments (`# (B, C, H, W)`), and use type hints. This demonstrates production-level habits and makes your code reviewable in real time.

**4. Know your shapes.** The most common source of bugs in diffusion code is shape mismatches. Practice saying shapes out loud: "betas is (T,), alpha_bars[t] is (B,), I need to unsqueeze to (B, 1, 1, 1) for broadcasting with images of shape (B, C, H, W)."

**5. Do not panic if you forget a formula.** If you cannot remember whether the reverse mean uses $\alpha_t$ or $\bar\alpha_t$, derive it from first principles. Say: "The reverse step removes one step of noise, so we need the cumulative noise level, which is alpha_bar." Interviewers respect derivation over memorization.

**6. Plan before coding.** Spend 2-3 minutes outlining your approach on paper or in comments before writing code. List the components you need (schedule, model, training loop, sampling) and their interfaces. This prevents the "write-delete-rewrite" cycle that wastes time.

**7. Know the math, but lead with intuition.** When asked a conceptual question, start with the intuitive answer ("we predict noise because it gives uniform difficulty across timesteps") and then offer to go deeper into the math. Most interviewers prefer clear intuition over symbol-heavy derivations -- but being able to do both is the strongest signal.